In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp>=2024.11.18',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
], check=True)

In [ ]:
import os
import json
import time
import threading
import subprocess
import sys
from pathlib import Path
from datetime import datetime

import yaml
import requests
from huggingface_hub import HfApi

WORK_DIR = Path('/kaggle/working')
MANIFEST_PATH = WORK_DIR / 'video_manifest.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1a.json'
QUERY_CONFIG = Path('/kaggle/input/urdu-asr-pipelines/config/query_bank.yaml')
HF_REPOS_CONFIG = Path('/kaggle/input/urdu-asr-pipelines/config/hf_repos.yaml')

WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'ANTHROPIC_API_KEY':  c.get_secret('ANTHROPIC_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass

    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')

    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY', 'ANTHROPIC_API_KEY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')

    return {k: os.environ[k] for k in required}

SECRETS = load_secrets()
HF_TOKEN = SECRETS['HF_TOKEN_PRIMARY']

In [ ]:
with open(QUERY_CONFIG) as f:
    query_cfg = yaml.safe_load(f)

with open(HF_REPOS_CONFIG) as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
HF_MANIFEST_FILENAME = 'video_manifest.jsonl'
HF_CHECKPOINT_FILENAME = 'checkpoint_p1a.json'

MIN_DUR = query_cfg['min_duration_sec']
MAX_DUR = query_cfg['max_duration_sec']
MAX_PER_QUERY = query_cfg['max_videos_per_query']
MAX_TOTAL = query_cfg['max_total_videos']

all_queries = []
for category, query_list in query_cfg['queries'].items():
    for q in query_list:
        all_queries.append({'query': q, 'category': category})

print(f'[config] {len(all_queries)} queries across {len(query_cfg["queries"])} categories')
print(f'[config] max {MAX_PER_QUERY} videos/query, {MAX_TOTAL} total')
print(f'[config] duration filter: {MIN_DUR}s — {MAX_DUR}s')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — queries_done={len(state["queries_done"])} videos_found={state["stats"]["total_found"]}')
            return state
        except Exception:
            pass

    try:
        api = HfApi(token=HF_TOKEN)
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/{HF_CHECKPOINT_FILENAME}'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — queries_done={len(state["queries_done"])}')
            return state
    except Exception:
        pass

    print('[checkpoint] fresh start')
    return {
        'queries_done': [],
        'seen_video_ids': [],
        'stats': {'total_found': 0, 'total_filtered': 0, 'queries_attempted': 0},
        'last_updated': None,
    }


def save_checkpoint(state, upload=True):
    state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    tmp = str(CHECKPOINT_PATH) + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(state, f)
    os.replace(tmp, str(CHECKPOINT_PATH))

    if not upload:
        return

    for attempt in range(6):
        try:
            api = HfApi(token=HF_TOKEN)
            api.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo=HF_CHECKPOINT_FILENAME,
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1a checkpoint',
            )
            return
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f'[checkpoint] upload failed attempt {attempt+1}: {e} — retry in {wait}s')
            time.sleep(wait)


state = load_checkpoint()
seen_video_ids = set(state['seen_video_ids'])

In [ ]:
def discover_videos_for_query(query, category, max_results, min_dur, max_dur):
    cmd = [
        'yt-dlp',
        '--flat-playlist',
        '--no-warnings',
        '--print', '%(.{id,title,duration,channel_id,channel,view_count,upload_date})j',
        '--match-filter', f'duration >= {min_dur} & duration <= {max_dur}',
        f'ytsearch{max_results}:{query}',
    ]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=120,
        )
    except subprocess.TimeoutExpired:
        print(f'  [timeout] query: {query}')
        return []
    except Exception as e:
        print(f'  [error] query: {query} — {e}')
        return []

    videos = []
    for line in result.stdout.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            info = json.loads(line)
        except json.JSONDecodeError:
            continue

        vid_id = info.get('id')
        duration = info.get('duration')

        if not vid_id or not duration:
            continue
        if not (min_dur <= duration <= max_dur):
            continue

        videos.append({
            'video_id': vid_id,
            'title': info.get('title', ''),
            'channel_id': info.get('channel_id', ''),
            'channel_name': info.get('channel', ''),
            'duration_sec': duration,
            'view_count': info.get('view_count', 0),
            'upload_date': info.get('upload_date', ''),
            'query_used': query,
            'category': category,
            'discovered_at': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
        })

    return videos

In [ ]:
pending_queries = [
    q for q in all_queries
    if q['query'] not in state['queries_done']
]

print(f'[discovery] {len(pending_queries)} queries remaining, {len(state["queries_done"])} already done')
print(f'[discovery] {len(seen_video_ids)} videos already in manifest')

SAVE_EVERY = 5

for idx, q_item in enumerate(pending_queries):
    if state['stats']['total_found'] >= MAX_TOTAL:
        print(f'[discovery] reached MAX_TOTAL={MAX_TOTAL}, stopping')
        break

    query = q_item['query']
    category = q_item['category']

    print(f'  [{idx+1}/{len(pending_queries)}] {query}')

    for attempt in range(3):
        videos = discover_videos_for_query(query, category, MAX_PER_QUERY, MIN_DUR, MAX_DUR)
        if videos or attempt == 2:
            break
        time.sleep(5 * (attempt + 1))

    new_count = 0
    with open(MANIFEST_PATH, 'a', encoding='utf-8') as mf:
        for v in videos:
            if v['video_id'] in seen_video_ids:
                state['stats']['total_filtered'] += 1
                continue
            seen_video_ids.add(v['video_id'])
            mf.write(json.dumps(v, ensure_ascii=False) + '\n')
            new_count += 1

    state['queries_done'].append(query)
    state['seen_video_ids'] = list(seen_video_ids)
    state['stats']['total_found'] += new_count
    state['stats']['total_filtered'] += len(videos) - new_count
    state['stats']['queries_attempted'] += 1

    print(f'     found={len(videos)} new={new_count} total={state["stats"]["total_found"]} dupes={state["stats"]["total_filtered"]}')

    upload_now = (idx + 1) % SAVE_EVERY == 0
    save_checkpoint(state, upload=upload_now)

    time.sleep(2)

In [ ]:
total_in_manifest = sum(1 for _ in open(MANIFEST_PATH, encoding='utf-8')) if MANIFEST_PATH.exists() else 0
print(f'[done] manifest has {total_in_manifest} videos')
print(f'[done] queries completed: {len(state["queries_done"])}/{len(all_queries)}')
print(f'[done] duplicates filtered: {state["stats"]["total_filtered"]}')

save_checkpoint(state, upload=True)

for attempt in range(8):
    try:
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=str(MANIFEST_PATH),
            path_in_repo=HF_MANIFEST_FILENAME,
            repo_id=STAGE0_REPO,
            repo_type='dataset',
            commit_message=f'p1a manifest — {total_in_manifest} videos',
        )
        print(f'[done] manifest uploaded to {STAGE0_REPO}')
        break
    except Exception as e:
        wait = min(2 ** attempt, 60)
        print(f'[upload] attempt {attempt+1} failed: {e} — retry in {wait}s')
        time.sleep(wait)